# Heart Sound Analysis -- Automated Benchmark & Audit Pipeline
### Leakage-Aware Evaluation, 5 Architectural Ablations, Explanation Audits & Paper Deliverables

**Instructions:**
1. Go to **Runtime -> Change runtime type** and ensure **T4 GPU** is selected.
2. Click **Runtime -> Run all** (or press `Ctrl + F9`).
3. The notebook will automatically execute the entire 24-experiment benchmark, audit explanation faithfulness, compile all LaTeX tables and macros, and download the results.


## 0. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability (CUDA or Apple Silicon MPS)
import sys
import shutil
import torch

print(f'PyTorch Version: {torch.__version__}')
cuda_avail = torch.cuda.is_available()
mps_avail = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
print(f'CUDA Available: {cuda_avail}')
print(f'Apple Silicon MPS Available: {mps_avail}')

if cuda_avail:
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
    if shutil.which('nvidia-smi'):
        !nvidia-smi
elif mps_avail:
    print('Using Apple Silicon Metal Performance Shaders (MPS) GPU acceleration.')
else:
    print('Running on CPU.')


In [ ]:
# Install required Python dependencies
%pip install -q librosa scipy numpy pandas torch torchvision tqdm shap h5py pytest

## 1. Repository Setup & Drive Mount (Optional)

In [ ]:
import os
import subprocess
from pathlib import Path

# 1. Mount Google Drive if in Colab
is_colab = os.path.exists('/content')
if is_colab:
    if not Path('/content/drive/MyDrive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    print('Google Drive mounted.')
    
    # 2. Always use fresh GitHub code in /content/heart-sound-analysis
    repo_dir = Path('/content/heart-sound-analysis')
    if not repo_dir.exists():
        print('Cloning latest codebase from GitHub...')
        !git clone https://github.com/urvagandhi/heart-sound-analysis.git /content/heart-sound-analysis
    else:
        print('Pulling latest updates from GitHub...')
        !git -C /content/heart-sound-analysis pull
        
    # 3. Locate dataset on Google Drive
    drive_candidates = [
        Path('/content/drive/MyDrive/RMS'),
        Path('/content/drive/MyDrive/CSE/College/RMS'),
    ]
    drive_rms = next((d for d in drive_candidates if d.exists()), None)
    
    if drive_rms and (drive_rms / 'physionet_2016').exists():
        drive_data = drive_rms / 'physionet_2016'
        target_data = repo_dir / 'physionet_2016'
        if not target_data.exists():
            print(f'Linking dataset: {drive_data} -> {target_data}')
            os.symlink(str(drive_data), str(target_data))
        else:
            print(f'Dataset already accessible at {target_data}')
            
        # 4. Link results directory to Drive so all training runs persist permanently
        drive_results = drive_rms / 'results'
        drive_results.mkdir(parents=True, exist_ok=True)
        target_results = repo_dir / 'results'
        if not target_results.exists():
            print(f'Linking persistent results: {drive_results} -> {target_results}')
            os.symlink(str(drive_results), str(target_results))
        else:
            print(f'Results folder active at {target_results}')
    else:
        print('Warning: physionet_2016 not found in Drive RMS folder. Using repo files.')
else:
    print('Running in local environment (MacBook).')
    repo_dir = Path.cwd()

os.chdir(str(repo_dir))
print(f'Active working directory: {Path.cwd()}')


## 2. Dataset Verification & Extraction

In [ ]:
import zipfile
data_dir = Path("physionet_2016")
raw_dir = data_dir / "raw"
spec_dir = data_dir / "spectrograms"
zip_file = Path("physionet_2016.zip")

# Extract if zip exists and spectrograms not yet unpacked
if zip_file.exists() and (not spec_dir.exists() or len(list(spec_dir.glob("*.npy"))) < 100):
    print("Extracting physionet_2016.zip into physionet_2016/...")
    with zipfile.ZipFile(zip_file, "r") as z:
        for m in z.infolist():
            rel_name = m.filename.split("physionet_2016/")[-1]
            if not rel_name:
                continue
            target = data_dir / rel_name
            if m.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, "wb") as dst:
                    dst.write(src.read())
    print("Extraction complete!")

print(f"Spectrograms available: {len(list(spec_dir.glob('*.npy')))} / 3541")
print(f"Metadata CSV exists: {(data_dir / 'metadata.csv').exists()}")

## 3. Generate Splits & Run Duplicate Leakage Audit

In [ ]:
# Generate deduplicated (3 seeds) and LODO splits
import sys
py_exec = sys.executable

!{py_exec} 'Review 3/make_splits.py' --mode dedup --seeds 42 43 44
!{py_exec} 'Review 3/make_splits.py' --mode lodo --holdouts a b e f --seed 42

# Check twin duplicate performance vs unique test accuracy
!{py_exec} 'Review 3/verify_duplicates.py'


## 4. Run All 24 Benchmark Experiments Sequentially

In [ ]:
# Dry-run verification (24 planned runs)
import sys
py_exec = sys.executable
!{py_exec} 'Review 3/run_all.py' --dry-run

# Execute full 24-run benchmark (resume-safe: skips already completed runs)
!{py_exec} 'Review 3/run_all.py'


## 5. Run Population-Level Explanation Audit Suite

In [ ]:
# Execute audit on primary model: dedup_se_mha_seed42
import sys
py_exec = sys.executable
!{py_exec} 'Review 3/audit/run_audit.py' --tag dedup_se_mha_seed42


## 6. Aggregate Deliverables & Compile LaTeX Paper

In [ ]:
# Aggregate all JSON artifacts into LaTeX tables, macros, and findings
import sys, shutil
py_exec = sys.executable
!{py_exec} 'Review 3/aggregate_results.py'

# Run unit tests to verify consistency
!{py_exec} -m pytest -q

# Compile PDF using pdflatex if available
if shutil.which('pdflatex'):
    %cd 'Paper Writing'
    !pdflatex -interaction=nonstopmode paper.tex
    !bibtex paper
    !pdflatex -interaction=nonstopmode paper.tex
    !pdflatex -interaction=nonstopmode paper.tex
    %cd ..
elif shutil.which('apt-get'):
    !apt-get update -qq && apt-get install -y -qq texlive-latex-base texlive-fonts-recommended texlive-fonts-extra texlive-latex-extra
    %cd 'Paper Writing'
    !pdflatex -interaction=nonstopmode paper.tex
    !bibtex paper
    !pdflatex -interaction=nonstopmode paper.tex
    !pdflatex -interaction=nonstopmode paper.tex
    %cd ..
else:
    print('pdflatex not found on system. Skipping PDF compilation.')


## 7. Download Results Archive

In [ ]:
from pathlib import Path

# Create zip of all experimental results and the compiled paper PDF
output_archive = 'benchmark_results_and_paper.zip'
!zip -q -r benchmark_results_and_paper.zip results 'Paper Writing/generated' 
print(f'Created archive: {output_archive}')

try:
    from google.colab import files
    files.download(output_archive)
except Exception:
    print(f'Saved locally at: {Path.cwd() / output_archive}')
